<a href="https://colab.research.google.com/github/dklishta/python-ai-Gailunaite-Darya/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- [`cartoons_genre_country_duration.csv`](https://github.com/componavt/python-ai-template/blob/main/data/examples/cartoons_genre_country_duration.sparql) — жанры, страны и продолжительность мультфильмов
- [`cartoons_assessment_reviews.csv`](https://github.com/componavt/python-ai-template/blob/main/data/examples/cartoons_assessment_reviews.sparql) — оценки и рецензии мультфильмов

**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [ ]:
# 🐱 Шаг 1. Клонируем ваш репозиторий курса в Colab

import os

repo = "python-ai-Gailunaite-Darya"  # ← ИЗМЕНЕНО: имя вашего репозитория
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Gailunaite-Darya
    !git clone -q https://github.com/dklishta/python-ai-Gailunaite-Darya.git  # ← ИЗМЕНЕНО: URL вашего репозитория

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Gailunaite-Darya


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [ ]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

df_teacoffee = pd.read_csv("data/tea_coffee_info.csv")
df_teacoffeefin = pd.read_csv("data/tea_coffee_fin.csv")

print("✅ Загружено строк в df_teacoffee:", len(df_teacoffee))
print("✅ Загружено строк в df_teacoffeefin:", len(df_teacoffeefin))

✅ Загружено строк в df_teacoffee: 289
✅ Загружено строк в df_teacoffeefin: 1555


## 🧹 [2B] Очистка и переименование столбцов

В вашем CSV-файле `tea_coffee_info.csv` есть **технические столбцы**, которые требуют обработки:

- Столбец `company` с URL (ссылкой на объект Wikidata) — **удаляем**, так как для анализа он не нужен.
- Столбцы с суффиксом `Label` (`companyLabel`, `productTypeLabel`, `countryLabel`, `hqLabel`) содержат читаемые названия — **переименуем их**, убрав постфикс `Label`.
- Столбец `hqCoord` содержит координаты в формате WKT — **оставляем как есть** (для возможной визуализации на карте).
- Столбец `foundingYearShort` содержит год основания — **приведём к числовому типу** для фильтрации и сортировки.

В этом шаге мы:
- **удаляем** столбец `company` (URL Wikidata);
- **переименовываем**: `companyLabel → company`, `productTypeLabel → productType`, `countryLabel → country`, `hqLabel → hq`;
- **приводим** `foundingYearShort` к типу `int` (целое число).

При приведении к числам мы используем:
- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные → `NaN`;
- `fillna(0)` — заменяет пропуски на 0;
- `astype(int)` — переводит столбец к целочисленному типу.

> ⚠️ **Важно:** после этого шага у вас останется аккуратная таблица с понятными названиями столбцов, готовая к фильтрации, группировке и визуализации.

In [12]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для tea_coffee_info.csv
# (с защитой от NameError и улучшенной загрузкой данных)

import pandas as pd
import os

# 🔁 Если df не определён — пробуем загрузить файл заново
if 'df' not in globals():
    print("⚠️ Переменная 'df' не найдена. Пробуем загрузить данные...")

    file_path = "data/tea_coffee_info.csv"

    # Проверяем существование файла
    if not os.path.exists(file_path):
        # Пробуем альтернативные пути (частая проблема в Colab)
        alt_paths = [
            "/content/python-ai-Gailunaite-Darya/data/tea_coffee_info.csv",
            "tea_coffee_info.csv",
            "../data/tea_coffee_info.csv"
        ]
        for alt in alt_paths:
            if os.path.exists(alt):
                file_path = alt
                print(f"✅ Файл найден по альтернативному пути: {file_path}")
                break
        else:
            raise FileNotFoundError(
                f"❌ Файл не найден!\n"
                f"Текущая папка: {os.getcwd()}\n"
                f"Ожидаемый путь: data/tea_coffee_info.csv\n"
                f"Содержимое текущей папки: {os.listdir('.')}"
            )

    # Пробуем разные кодировки и разделители
    df = None
    for encoding in ["utf-8", "cp1251", "utf-8-sig"]:
        for sep in [",", ";", "\t"]:
            try:
                df = pd.read_csv(file_path, encoding=encoding, sep=sep)
                print(f"✅ Файл успешно загружен: encoding='{encoding}', sep='{sep}'")
                break
            except Exception:
                continue
        if df is not None:
            break

    if df is None:
        raise ValueError(
            "❌ Не удалось прочитать CSV-файл.\n"
            "Проверьте: кодировку файла, разделитель, целостность данных."
        )

# ============================================================
# 🧹 Основная логика очистки и переименования
# ============================================================

# 1) Удаляем технический столбец с URL Wikidata (если существует)
if "company" in df.columns:
    df = df.drop(columns=["company"])
    print("✅ Столбец 'company' (URL) удалён")
else:
    print("⏭️ Столбец 'company' не найден, пропускаем удаление")

# 2) Переименовываем столбцы: убираем суффикс Label
rename_map = {
    "companyLabel": "company",
    "productTypeLabel": "productType",
    "countryLabel": "country",
    "hqLabel": "hq",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
print("✅ Столбцы переименованы:", list(df.columns))

# 3) Приводим foundingYearShort к числовому типу (если столбец существует)
if "foundingYearShort" in df.columns:
    df["foundingYearShort"] = pd.to_numeric(
        df["foundingYearShort"], errors="coerce"
    ).fillna(0).astype(int)
    print("✅ foundingYearShort приведён к типу int")
else:
    print("⏭️ Столбец foundingYearShort не найден, пропускаем преобразование")

# 4) Показываем результат
print("\n📋 Первые 5 строк после очистки:")
display(df.head())

print(f"\n📊 Информация о DataFrame: {df.shape[0]} строк, {df.shape[1]} столбцов")
print("\n✅ Данные готовы к анализу")

⏭️ Столбец 'company' не найден, пропускаем удаление
✅ Столбцы переименованы: ['productType', 'country', 'hq', 'hqCoord', 'foundingYearShort']
✅ foundingYearShort приведён к типу int

📋 Первые 5 строк после очистки:


,productType,country,hq,hqCoord,foundingYearShort
0,чай,Соединённое королевство Великобритании и Ирландии,Лондон,Point(-0.1275 51.507222222),1600
1,чай,Соединённое королевство Великобритании и Ирландии,Лондон,Point(-0.1275 51.507222222),1600
2,кофе,Канада,Оквилл,Point(-79.683333333 43.45),1964
3,кофе,Швейцария,Лозанна,Point(6.633333333 46.533333333),1986
4,кофе,США,Нортфилд,Point(-87.766666666 42.1),2012



📊 Информация о DataFrame: 289 строк, 5 столбцов

✅ Данные готовы к анализу


In [18]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для tea_coffee_fin.csv
# (с защитой от NameError и авто-загрузкой данных)

import pandas as pd
import os

print("🧹 ОЧИСТКА ФИНАНСОВЫХ ДАННЫХ")
print("=" * 70)

# ============================================================
# 🔁 0. Если df_fin не определён — пробуем загрузить файл заново
# ============================================================

if 'df_fin' not in globals():
    print("⚠️  Переменная 'df_fin' не найдена. Пробуем загрузить данные...")

    file_path = "data/tea_coffee_fin.csv"

    # Проверяем существование файла
    if not os.path.exists(file_path):
        alt_paths = [
            "/content/python-ai-Gailunaite-Darya/data/tea_coffee_fin.csv",
            "tea_coffee_fin.csv",
            "../data/tea_coffee_fin.csv"
        ]
        for alt in alt_paths:
            if os.path.exists(alt):
                file_path = alt
                print(f"✅ Файл найден по альтернативному пути: {file_path}")
                break
        else:
            raise FileNotFoundError(
                f"❌ Файл не найден!\n"
                f"Текущая папка: {os.getcwd()}\n"
                f"Ожидаемый путь: {file_path}\n"
                f"Содержимое папки: {os.listdir('.')}"
            )

    # Пробуем разные кодировки и разделители
    df_fin = None
    for encoding in ["utf-8", "cp1251", "utf-8-sig"]:
        for sep in [",", ";", "\t"]:
            try:
                df_fin = pd.read_csv(file_path, encoding=encoding, sep=sep)
                print(f"✅ Файл успешно загружен: encoding='{encoding}', sep='{sep}'")
                break
            except Exception as e:
                continue
        if df_fin is not None:
            break

    if df_fin is None:
        raise ValueError(
            "❌ Не удалось прочитать CSV-файл.\n"
            "Проверьте: кодировку, разделитель, целостность файла."
        )

# ============================================================
# 🧹 1) Удаляем технический столбец с URL
# ============================================================

if "company" in df_fin.columns:
    df_fin = df_fin.drop(columns=["company"])
    print("✅ Столбец 'company' (URL) удалён")
else:
    print("⏭️  Столбец 'company' не найден, пропускаем удаление")

# ============================================================
# 🧹 2) Переименовываем столбцы: убираем суффикс Label
# ============================================================

rename_map = {
    "companyLabel": "company",
    "productTypeLabel": "productType",
    "revenueCurrencyLabel": "revenueCurrency",
    "netProfitCurrencyLabel": "netProfitCurrency",
}
df_fin = df_fin.rename(columns={k: v for k, v in rename_map.items() if k in df_fin.columns})
print("✅ Столбцы переименованы:", list(df_fin.columns))

# ============================================================
# 🧹 3) Приводим финансовые показатели к числовому типу
# ============================================================

numeric_cols = ["revenue", "revenueYear", "employees", "employeesYear", "netProfit", "netProfitYear"]

for col in numeric_cols:
    if col in df_fin.columns:
        original_na = df_fin[col].isna().sum()
        df_fin[col] = pd.to_numeric(df_fin[col], errors="coerce")
        new_na = df_fin[col].isna().sum()
        converted = new_na - original_na
        if converted > 0:
            print(f"⚠️  '{col}': {converted} значений преобразовано в NaN (некорректный формат)")
        else:
            print(f"✅ '{col}' приведён к числовому типу")

# ============================================================
# 📊 4) Отчёт о заполненности финансовых данных
# ============================================================

print("\n📊 ЗАПОЛНЕННОСТЬ ФИНАНСОВЫХ ПОКАЗАТЕЛЕЙ")
print("-" * 70)
total = len(df_fin)
for col in ["revenue", "employees", "netProfit"]:
    if col in df_fin.columns:
        filled = df_fin[col].notna().sum()
        pct = filled / total * 100
        print(f"{col:<15} {filled:>4} из {total} записей ({pct:>5.1f}%)")

print("\n✅ Данные очищены и готовы к анализу")

🧹 ОЧИСТКА ФИНАНСОВЫХ ДАННЫХ
⚠️  Переменная 'df_fin' не найдена. Пробуем загрузить данные...
✅ Файл успешно загружен: encoding='utf-8', sep=','
✅ Столбец 'company' (URL) удалён
✅ Столбцы переименованы: ['company', 'productType', 'revenue', 'revenueYear', 'revenueCurrency', 'employees', 'employeesYear', 'netProfit', 'netProfitYear', 'netProfitCurrency']
✅ 'revenue' приведён к числовому типу
✅ 'revenueYear' приведён к числовому типу
✅ 'employees' приведён к числовому типу
✅ 'employeesYear' приведён к числовому типу
✅ 'netProfit' приведён к числовому типу
✅ 'netProfitYear' приведён к числовому типу

📊 ЗАПОЛНЕННОСТЬ ФИНАНСОВЫХ ПОКАЗАТЕЛЕЙ
----------------------------------------------------------------------
revenue         1318 из 1555 записей ( 84.8%)
employees       1326 из 1555 записей ( 85.3%)
netProfit       1310 из 1555 записей ( 84.2%)

✅ Данные очищены и готовы к анализу


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с данными о чайных и кофейных компаниях:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по **году основания** (`foundingYearShort`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы выводить информацию в структурированном виде.

**Ваши столбцы после очистки:**
- `company` — название компании
- `productType` — тип продукта (чай, кофе и т.д.)
- `country` — страна происхождения
- `hq` — город штаб-квартиры
- `hqCoord` — координаты штаб-квартиры (WKT-формат)
- `foundingYearShort` — год основания (число)

In [13]:
# 🔍 Шаг 3. Обзор данных

def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("=" * 60)
    print("Размер:", df.shape, f"({df.shape[0]} строк, {df.shape[1]} столбцов)")
    print("Столбцы:", ", ".join(df.columns))
    print("\n📋 Первые строки:")
    display(df.head(n))

    # 📈 Базовая статистика по числовым столбцам
    numeric_cols = df.select_dtypes(include=['int', 'float']).columns
    if len(numeric_cols) > 0:
        print("\n📈 Статистика по числовым столбцам:")
        print(df[numeric_cols].describe())
    else:
        print("\n⚠️ Нет числовых столбцов для статистики")

    # 🏷️ Уникальные значения категориальных столбцов
    print("\n🏷️ Уникальные значения:")
    for col in df.select_dtypes(include=['object']).columns:
        unique_count = df[col].nunique()
        print(f"  • {col}: {unique_count} уникальных значений")

# ============================================================
# 🔍 Вызов функции для вашего DataFrame
# ============================================================

show_info(df, "Чайные и кофейные компании (df)")

# ============================================================
# 💡 Дополнительно: быстрая сводка по вашим данным
# ============================================================

print("\n" + "=" * 60)
print("📌 БЫСТРАЯ СВОДКА ПО ДАННЫМ")
print("=" * 60)

if "productType" in df.columns:
    print("\n☕ Типы продуктов:")
    print(df["productType"].value_counts())

if "country" in df.columns:
    print("\n🌍 Страны:")
    print(df["country"].value_counts())

if "foundingYearShort" in df.columns:
    print("\n📅 Год основания:")
    print(f"  • Самый ранний: {df['foundingYearShort'].min()}")
    print(f"  • Самый поздний: {df['foundingYearShort'].max()}")
    print(f"  • Средний: {df['foundingYearShort'].mean():.0f}")

print("\n✅ Обзор данных завершён")


📊 Чайные и кофейные компании (df)
Размер: (289, 5) (289 строк, 5 столбцов)
Столбцы: productType, country, hq, hqCoord, foundingYearShort

📋 Первые строки:


,productType,country,hq,hqCoord,foundingYearShort
0,чай,Соединённое королевство Великобритании и Ирландии,Лондон,Point(-0.1275 51.507222222),1600
1,чай,Соединённое королевство Великобритании и Ирландии,Лондон,Point(-0.1275 51.507222222),1600
2,кофе,Канада,Оквилл,Point(-79.683333333 43.45),1964
3,кофе,Швейцария,Лозанна,Point(6.633333333 46.533333333),1986
4,кофе,США,Нортфилд,Point(-87.766666666 42.1),2012



📈 Статистика по числовым столбцам:
       foundingYearShort
count         289.000000
mean         1612.733564
std           732.986245
min             0.000000
25%          1845.000000
50%          1933.000000
75%          1993.000000
max          2022.000000

🏷️ Уникальные значения:
  • productType: 2 уникальных значений
  • country: 50 уникальных значений
  • hq: 145 уникальных значений
  • hqCoord: 150 уникальных значений

📌 БЫСТРАЯ СВОДКА ПО ДАННЫМ

☕ Типы продуктов:
productType
кофе    178
чай     111
Name: count, dtype: int64

🌍 Страны:
country
США                                                  41
Нидерланды                                           24
Великобритания                                       18
Канада                                               11
Соединённое королевство Великобритании и Ирландии    10
Австралия                                             9
Индия                                                 7
Дания                                             

In [19]:
# 🔍 Шаг 3. Обзор финансовых данных

def show_fin_info(df, name, n=5):
    """Расширенный обзор финансового DataFrame."""
    print(f"\n📊 {name}")
    print("=" * 70)
    print(f"Размер: {df.shape[0]} строк × {df.shape[1]} столбцов")
    print("Столбцы:", ", ".join(df.columns))

    print("\n📋 Первые строки:")
    display(df.head(n))

    # Статистика по числовым столбцам
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print("\n📈 Статистика по финансовым показателям:")
        display(df[numeric_cols].describe().round(2))

    # Категориальные столбцы
    print("\n🏷️ Категориальные данные:")
    for col in df.select_dtypes(include=['object']).columns:
        unique = df[col].nunique()
        top = df[col].mode()[0] if not df[col].mode().empty else "N/A"
        print(f"• {col}: {unique} уникальных | чаще всего: '{top}'")

# Запуск обзора
show_fin_info(df_fin, "Финансовые показатели чайно-кофейных компаний")

# ============================================================
# 💡 Дополнительные быстрые срезы
# ============================================================

print("\n" + "=" * 70)
print("📌 БЫСТРЫЕ СРЕЗЫ ПО ДАННЫМ")
print("=" * 70)

# Продукты
if "productType" in df_fin.columns:
    print("\n☕ Распределение по типам продуктов:")
    print(df_fin["productType"].value_counts())

# Валюты
currency_cols = ["revenueCurrency", "netProfitCurrency"]
for col in currency_cols:
    if col in df_fin.columns and df_fin[col].notna().any():
        print(f"\n💱 Уникальные валюты в {col}:")
        print(df_fin[col].dropna().unique())

# Компании с данными по выручке
if "revenue" in df_fin.columns and "company" in df_fin.columns:
    with_revenue = df_fin[df_fin["revenue"].notna()]
    if len(with_revenue) > 0:
        print(f"\n💰 Компании с данными по выручке ({len(with_revenue)} шт.):")
        display(with_revenue[["company", "revenue", "revenueYear", "revenueCurrency"]].head())

print("\n✅ Обзор завершён")


📊 Финансовые показатели чайно-кофейных компаний
Размер: 1555 строк × 10 столбцов
Столбцы: company, productType, revenue, revenueYear, revenueCurrency, employees, employeesYear, netProfit, netProfitYear, netProfitCurrency

📋 Первые строки:


,company,productType,revenue,revenueYear,revenueCurrency,employees,employeesYear,netProfit,netProfitYear,netProfitCurrency
0,Q101249024,чай,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bünting-Gruppe,чай,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Tjiboeni-Tjipongpok Caoutchouc Maatschappij,кофе,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Tjiboeni-Tjipongpok Caoutchouc Maatschappij,чай,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Sumatra Rubber Cultuur Maatschappij ""Serbadjadi""",кофе,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



📈 Статистика по финансовым показателям:


,revenue,revenueYear,employees,employeesYear,netProfit,netProfitYear
count,1.318000e+03,1318.00,1326.00,1323.00,1.310000e+03,1309.00
mean,2.152226e+10,2017.02,239374.16,2017.76,2.066543e+09,2015.29
std,9.209741e+09,5.47,102704.71,3.19,1.372751e+09,5.98
min,2.033000e+06,2007.00,18.00,2014.00,-1.480300e+07,2000.00
25%,1.329950e+10,2012.00,191000.00,2014.00,9.283000e+08,2011.00
50%,2.351800e+10,2018.00,245000.00,2016.00,1.856400e+09,2015.00
75%,2.766100e+10,2022.00,383000.00,2021.00,3.281600e+09,2020.00
max,3.718440e+10,2025.00,383000.00,2024.00,4.518300e+09,2025.00



🏷️ Категориальные данные:
• company: 234 уникальных | чаще всего: 'Starbucks'
• productType: 2 уникальных | чаще всего: 'кофе'
• revenueCurrency: 5 уникальных | чаще всего: 'доллар США'
• netProfitCurrency: 4 уникальных | чаще всего: 'доллар США'

📌 БЫСТРЫЕ СРЕЗЫ ПО ДАННЫМ

☕ Распределение по типам продуктов:
productType
кофе    1457
чай       98
Name: count, dtype: int64

💱 Уникальные валюты в revenueCurrency:
['евро' 'чешская крона' 'Белорусский рубль' 'датская крона' 'доллар США']

💱 Уникальные валюты в netProfitCurrency:
['чешская крона' 'Белорусский рубль' 'евро' 'доллар США']

💰 Компании с данными по выручке (1318 шт.):


,company,revenue,revenueYear,revenueCurrency
34,Devolli Corporation,400000000.0,2021.0,евро
50,Jemča,152244000.0,2019.0,чешская крона
73,Белкофе,2033000.0,2023.0,Белорусский рубль
92,Mikel Coffee Company,33439000.0,2014.0,евро
93,Mikel Coffee Company,33439000.0,2014.0,евро



✅ Обзор завершён


## 🌍 [4.3] Географический рейтинг: какие страны лидируют?

В этом блоке мы отвечаем на вопрос: **из каких стран родом большинство компаний**, представленных в нашем датасете?

### 🔎 Что мы анализируем:
- **Топ-10 стран** по количеству записей о компаниях;
- **Долю каждой страны** в процентах от общего числа записей;
- **Разбивку по типу продукта** (чай / кофе) внутри топ-стран;
- **Накопленную долю** (cumulative share) — сколько процентов данных покрывает топ-N стран.

### 📊 Почему это важно:
- Показывает исторические центры чайной и кофейной торговли;
- Помогает выявить географические «слепые зоны» (регионы, слабо представленные в Викиданных);
- Даёт основу для дальнейшей фильтрации: например, «показать только европейские компании».

### 🧮 Методы pandas, которые мы используем:
| Метод | Зачем нужен |
|-------|-------------|
| `value_counts()` | Считает частоту каждого значения в столбце |
| `normalize=True` | Возвращает доли вместо абсолютных чисел |
| `cumsum()` | Считает накопленную сумму (для cumulative share) |
| `groupby() + size()` | Группирует данные по нескольким столбцам |
| `unstack()` | Превращает группировку в сводную таблицу |

> 💡 **Подсказка**: если в столбце `country` есть пропуски (`NaN`), они автоматически исключаются из `value_counts()`. Мы явно проверим это и сообщим о пропущенных значениях.

In [16]:
# 🌍 Шаг 4.3. Географический рейтинг: какие страны лидируют?

import pandas as pd

print("🌍 АНАЛИЗ: Географическое распределение компаний")
print("=" * 70)

# ============================================================
# 🔍 0. Предварительная проверка данных
# ============================================================

if "country" not in df.columns:
    raise ValueError("❌ Столбец 'country' не найден в DataFrame!\n"
                     f"Доступные столбцы: {list(df.columns)}")

# Проверяем пропуски в столбце country
missing_countries = df["country"].isna().sum()
total_rows = len(df)
print(f"📋 Всего записей: {total_rows}")
print(f"⚠️  Записей без указания страны: {missing_countries} "
      f"({missing_countries/total_rows*100:.1f}%)")

# Фильтруем данные: оставляем только строки с указанной страной
df_countries = df[df["country"].notna()].copy()
print(f"✅ Записей для анализа: {len(df_countries)}\n")

# ============================================================
# 📊 1. Топ-10 стран по количеству компаний
# ============================================================

print("🏆 ТОП-10 СТРАН ПО КОЛИЧЕСТВУ КОМПАНИЙ")
print("-" * 70)

country_counts = df_countries["country"].value_counts()
country_pct = df_countries["country"].value_counts(normalize=True) * 100
country_cumulative = country_pct.cumsum()

# Создаём сводную таблицу для вывода
top_n = 10
summary = pd.DataFrame({
    "Страна": country_counts.index[:top_n],
    "Количество": country_counts.values[:top_n],
    "Доля, %": country_pct.values[:top_n],
    "Накопленная доля, %": country_cumulative.values[:top_n]
})

# Форматируем вывод
for _, row in summary.iterrows():
    bar = "█" * int(row["Доля, %"] / 2)  # мини-гистограмма текстом
    print(f"{row['Страна']:<40} {row['Количество']:>3} | {row['Доля, %']:>5.1f}% {bar}")

print(f"\n📈 Топ-{top_n} стран покрывают {country_cumulative.iloc[min(top_n-1, len(country_cumulative)-1)]:.1f}% всех записей")

# ============================================================
# ☕🆚🫖 2. Разбивка по типу продукта внутри топ-5 стран
# ============================================================

if "productType" in df_countries.columns:
    print("\n\n☕🆚🫖 РАЗБИВКА ПО ПРОДУКТАМ В ТОП-5 СТРАНАХ")
    print("-" * 70)

    top_5_countries = country_counts.index[:5]
    df_top5 = df_countries[df_countries["country"].isin(top_5_countries)]

    # Сводная таблица: страна × продукт
    product_pivot = df_top5.groupby(["country", "productType"]).size().unstack(fill_value=0)

    # Добавляем итоговый столбец
    product_pivot["Всего"] = product_pivot.sum(axis=1)

    # Сортируем по убыванию и выводим
    product_pivot = product_pivot.sort_values("Всего", ascending=False)

    print(product_pivot.to_markdown(index=True, tablefmt="grid"))

# ============================================================
# 🧮 3. Дополнительные метрики
# ============================================================

print("\n\n📊 ДОПОЛНИТЕЛЬНЫЕ МЕТРИКИ")
print("-" * 70)

print(f"• Всего уникальных стран: {df_countries['country'].nunique()}")
print(f"• Средняя частота страны: {len(df_countries) / df_countries['country'].nunique():.2f} компаний/страна")
print(f"• Медианная частота: {df_countries['country'].value_counts().median():.0f} компаний")

# Страны с одной компанией (long tail)
single_country_count = (country_counts == 1).sum()
print(f"• Стран с только одной компанией: {single_country_count} "
      f"({single_country_count / len(country_counts) * 100:.1f}% от всех стран)")

# ============================================================
# 💾 4. (Опционально) Сохранение результата
# ============================================================

# Если хотите сохранить рейтинг для отчёта:
# country_report = pd.DataFrame({
#     "country": country_counts.index,
#     "count": country_counts.values,
#     "percent": country_pct.values
# }).round(2)
# country_report.to_csv("data/country_ranking.csv", index=False, encoding="utf-8-sig")
# print("\n✅ Рейтинг стран сохранён в data/country_ranking.csv")

print("\n✅ Географический анализ завершён")

🌍 АНАЛИЗ: Географическое распределение компаний
📋 Всего записей: 289
⚠️  Записей без указания страны: 70 (24.2%)
✅ Записей для анализа: 219

🏆 ТОП-10 СТРАН ПО КОЛИЧЕСТВУ КОМПАНИЙ
----------------------------------------------------------------------
США                                       41 |  18.7% █████████
Нидерланды                                24 |  11.0% █████
Великобритания                            18 |   8.2% ████
Канада                                    11 |   5.0% ██
Соединённое королевство Великобритании и Ирландии  10 |   4.6% ██
Австралия                                  9 |   4.1% ██
Индия                                      7 |   3.2% █
Дания                                      7 |   3.2% █
Германия                                   7 |   3.2% █
Франция                                    6 |   2.7% █

📈 Топ-10 стран покрывают 63.9% всех записей


☕🆚🫖 РАЗБИВКА ПО ПРОДУКТАМ В ТОП-5 СТРАНАХ
----------------------------------------------------------------------
+--

In [20]:
# 🏆 Шаг 4. Финансовый рейтинг компаний

import pandas as pd

print("🏆 ФИНАНСОВЫЙ РЕЙТИНГ КОМПАНИЙ")
print("=" * 70)

# ============================================================
# 🔍 0. Проверка наличия необходимых столбцов
# ============================================================

required = ["company", "productType"]
financial = ["revenue", "employees", "netProfit"]

for col in required:
    if col not in df_fin.columns:
        raise ValueError(f"❌ Обязательный столбец '{col}' не найден!")

print(f"✅ Все необходимые столбцы присутствуют")
print(f"📋 Всего компаний в датасете: {len(df_fin)}\n")

# ============================================================
# 📊 1. Заполненность финансовых показателей
# ============================================================

print("📊 ЗАПОЛНЕННОСТЬ ФИНАНСОВЫХ ДАННЫХ")
print("-" * 70)
for col in financial:
    if col in df_fin.columns:
        filled = df_fin[col].notna().sum()
        print(f"{col:<12}: {filled:>3} из {len(df_fin)} компаний ({filled/len(df_fin)*100:.1f}%)")

# ============================================================
# 💰 2. Топ компаний по выручке
# ============================================================

if "revenue" in df_fin.columns:
    print("\n\n💰 ТОП-10 КОМПАНИЙ ПО ВЫРУЧКЕ")
    print("-" * 70)

    rev_df = df_fin[df_fin["revenue"].notna()].copy()

    if len(rev_df) > 0:
        top_rev = rev_df.nlargest(10, "revenue")[["company", "revenue", "revenueYear", "revenueCurrency", "productType"]]
        for i, (_, row) in enumerate(top_rev.iterrows(), 1):
            currency = row["revenueCurrency"] if pd.notna(row["revenueCurrency"]) else "N/A"
            year = int(row["revenueYear"]) if pd.notna(row["revenueYear"]) else "N/A"
            print(f"{i:>2}. {row['company']:<40} {row['revenue']:>15,.0f} {currency} ({year})")
    else:
        print("⚠️  Нет данных по выручке для формирования рейтинга")

# ============================================================
# 👥 3. Топ компаний по количеству сотрудников
# ============================================================

if "employees" in df_fin.columns:
    print("\n\n👥 ТОП-10 КОМПАНИЙ ПО КОЛИЧЕСТВУ СОТРУДНИКОВ")
    print("-" * 70)

    emp_df = df_fin[df_fin["employees"].notna()].copy()

    if len(emp_df) > 0:
        top_emp = emp_df.nlargest(10, "employees")[["company", "employees", "employeesYear", "productType"]]
        for i, (_, row) in enumerate(top_emp.iterrows(), 1):
            year = int(row["employeesYear"]) if pd.notna(row["employeesYear"]) else "N/A"
            print(f"{i:>2}. {row['company']:<40} {row['employees']:>10,.0f} сотрудников ({year})")
    else:
        print("⚠️  Нет данных по сотрудникам для формирования рейтинга")

# ============================================================
# 📈 4. Топ компаний по чистой прибыли
# ============================================================

if "netProfit" in df_fin.columns:
    print("\n\n📈 ТОП-10 КОМПАНИЙ ПО ЧИСТОЙ ПРИБЫЛИ")
    print("-" * 70)

    profit_df = df_fin[df_fin["netProfit"].notna()].copy()

    if len(profit_df) > 0:
        top_profit = profit_df.nlargest(10, "netProfit")[["company", "netProfit", "netProfitYear", "netProfitCurrency", "productType"]]
        for i, (_, row) in enumerate(top_profit.iterrows(), 1):
            currency = row["netProfitCurrency"] if pd.notna(row["netProfitCurrency"]) else "N/A"
            year = int(row["netProfitYear"]) if pd.notna(row["netProfitYear"]) else "N/A"
            print(f"{i:>2}. {row['company']:<40} {row['netProfit']:>15,.0f} {currency} ({year})")
    else:
        print("⚠️  Нет данных по прибыли для формирования рейтинга")

# ============================================================
# ☕🆚🫖 5. Сравнение чая и кофе (если есть productType)
# ============================================================

if "productType" in df_fin.columns:
    print("\n\n☕🆚🫖 СРАВНЕНИЕ: ЧАЙ vs КОФЕ")
    print("-" * 70)

    comparison = {}
    for metric in ["revenue", "employees", "netProfit"]:
        if metric in df_fin.columns:
            comp = df_fin.groupby("productType")[metric].agg(["count", "mean", "median", "max"])
            comp = comp.rename(columns={"count": "записей", "mean": "среднее", "median": "медиана", "max": "максимум"})
            print(f"\n{metric.upper()}:")
            print(comp.round(2).to_markdown())

print("\n\n✅ Финансовый рейтинг завершён")

🏆 ФИНАНСОВЫЙ РЕЙТИНГ КОМПАНИЙ
✅ Все необходимые столбцы присутствуют
📋 Всего компаний в датасете: 1555

📊 ЗАПОЛНЕННОСТЬ ФИНАНСОВЫХ ДАННЫХ
----------------------------------------------------------------------
revenue     : 1318 из 1555 компаний (84.8%)
employees   : 1326 из 1555 компаний (85.3%)
netProfit   : 1310 из 1555 компаний (84.2%)


💰 ТОП-10 КОМПАНИЙ ПО ВЫРУЧКЕ
----------------------------------------------------------------------
 1. Starbucks                                 37,184,400,000 доллар США (2025)
 2. Starbucks                                 37,184,400,000 доллар США (2025)
 3. Starbucks                                 37,184,400,000 доллар США (2025)
 4. Starbucks                                 37,184,400,000 доллар США (2025)
 5. Starbucks                                 37,184,400,000 доллар США (2025)
 6. Starbucks                                 37,184,400,000 доллар США (2025)
 7. Starbucks                                 37,184,400,000 доллар США (2025)
 8. 

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали **ваш репозиторий** `python-ai-Gailunaite-Darya` в Google Colab
- ✅ Прочитали два CSV-файла из папки `data/`:
  - 🫖 `tea_coffee_info.csv` — общая информация о компаниях (страна, штаб-квартира, год основания)
  - 💰 `tea_coffee_fin.csv` — финансовые показатели (выручка, сотрудники, чистая прибыль)
- ✅ Очистили и нормализовали данные:
  - Удалили технические столбцы с URL Wikidata (`company`)
  - Переименовали столбцы с суффиксом `*Label` в короткие имена (`companyLabel` → `company`)
  - Привели числовые показатели (`foundingYearShort`, `revenue`, `employees`, `netProfit`) к типу `int/float`
  - Добавили защиту от ошибок кодировки и пропущенных значений
- ✅ Выполнили разведочный анализ (EDA):
  - 📊 Проверили размер таблиц, список столбцов, первые строки
  - 🔍 Оценили заполненность данных (% реальных значений в финансовых полях)
  - 🌍 Построили **географический рейтинг**: топ стран по количеству компаний
  - ☕🆚🫖 Сравняли распределение по типам продуктов (чай vs кофе)
  - 🏆 Сформировали **финансовый рейтинг**: топ компаний по выручке, сотрудникам и прибыли
- ✅ Написали вспомогательные функции:
  - `show_info()` / `show_fin_info()` — быстрый обзор DataFrame
  - `companies_by_country()` / `find_company()` — интерактивный поиск

---

### 🎯 Ключевые инсайты (на основе ваших данных):

| Вопрос | Ответ (пример) |
|--------|---------------|
| 🌍 Какие страны лидируют? | *Зависит от данных: например, Великобритания, Германия* |
| ☕ Чая или кофе больше? | *Сравните `df["productType"].value_counts()`* |
| 💰 У кого самая высокая выручка? | *Смотрите топ в ячейке финансового рейтинга* |
| 📅 Сколько компаний основано до 1800 года? | *Фильтр: `df[df["foundingYearShort"] < 1800]`* |
| ❓ Насколько полные финансовые данные? | *Проверьте отчёт о заполненности в ячейке очистки* |

> 💡 **Примечание**: Если какие-то рейтинги оказались пустыми — это не ошибка кода, а сигнал о том, что в Викиданных пока мало заполненных финансовых полей для этих компаний.

---

### 🚀 Что дальше? (Week 3 и далее)

В следующем ноутбуке мы сможем:

1. **Объединить два файла** (`info` + `fin`) по названию компании → получить единую таблицу `df_full`
2. **Углубить анализ**:
   - Группировка: «Средняя выручка по странам», «Динамика по десятилетиям»
   - Фильтрация: «Показать только чайные компании старше 100 лет»
   - Агрегация: «Общее количество сотрудников в индустрии»
3. **Построить визуализации** 🎨:
   - 🗺️ Карта штаб-квартир (по координатам `hqCoord`)
   - 📊 Столбчатая диаграмма: топ-10 компаний по выручке
   - 🥧 Круговая диаграмма: доля чая и кофе по странам
4. **Экспортировать результаты**:
   - Сохранить очищенные данные в `*_cleaned.csv`
   - Сгенерировать отчёт в Markdown или Excel

---

### 🧰 Полезные команды для самостоятельной работы:

```python
# 📋 Посмотреть уникальные значения
df_fin["productType"].unique()

# 🔍 Отфильтровать компании по условию
old_tea = df_info[(df_info["foundingYearShort"] < 1800) & (df_info["productType"] == "чай")]

# 📈 Быстрая статистика по группе
df_fin.groupby("productType")["revenue"].mean()

# 💾 Сохранить результат
df_fin.to_csv("data/tea_coffee_fin_cleaned.csv", index=False, encoding="utf-8-sig")